In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_3.py ──
"""
Shared infrastructure for MLFP02 Exercise 3 — A/B testing & multiple comparisons.

Contains: experiment data loading, group extraction, SRM check, common constants,
and small statistical helpers reused across the four technique files:

    01_bootstrap_power.py    — bootstrap CIs + MDE + power curves
    02_hypothesis_testing.py — two-proportion z-test + effect sizes
    03_multiple_testing.py   — Bonferroni + BH-FDR + FDR simulation
    04_permutation_test.py   — distribution-free alternative

Technique-specific code (the actual corrections, permutation loops, power
formulas) does NOT belong here — each technique file owns its own logic.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

ALPHA: float = 0.05
POWER_TARGET: float = 0.80
N_BOOTSTRAP: int = 10_000
N_PERMUTATIONS: int = 10_000
RANDOM_SEED: int = 42

OUTPUT_DIR = Path("outputs") / "mlfp02_ex3_ab_testing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce A/B test
# ════════════════════════════════════════════════════════════════════════


def load_experiment() -> pl.DataFrame:
    """Load the A/B test fixture used across Exercise 3.

    Columns: user_id, experiment_group, metric_value, pre_metric_value,
             revenue, timestamp, segment, platform, country.

    Derives a binary `converted` flag (metric_value > 0) if missing.
    Groups are binarised into {control, treatment}: anything that isn't
    literally "control" is treated as treatment (variant_c, treatment_a,
    etc.). This keeps the two-group tests simple for M2 pedagogy.
    """
    loader = MLFPDataLoader()
    df = loader.load("mlfp02", "experiment_data.parquet")

    if "converted" not in df.columns:
        df = df.with_columns(
            (pl.col("metric_value") > 0).cast(pl.Int8).alias("converted")
        )

    df = df.with_columns(
        pl.when(pl.col("experiment_group") == "control")
        .then(pl.lit("control"))
        .otherwise(pl.lit("treatment"))
        .alias("group")
    )
    return df


def split_groups(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Return (control_df, treatment_df)."""
    control = df.filter(pl.col("group") == "control")
    treatment = df.filter(pl.col("group") == "treatment")
    return control, treatment


def conversion_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_converted, treatment_converted) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["converted"].to_numpy().astype(np.float64)
    t = treatment["converted"].to_numpy().astype(np.float64)
    return c, t


def revenue_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_revenue, treatment_revenue) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["revenue"].to_numpy().astype(np.float64)
    t = treatment["revenue"].to_numpy().astype(np.float64)
    return c, t


# ════════════════════════════════════════════════════════════════════════
# SANITY CHECKS — SRM
# ════════════════════════════════════════════════════════════════════════


def srm_check(
    n_control: int, n_treatment: int, expected_ratio: float = 0.5
) -> dict[str, Any]:
    """χ² goodness-of-fit test for Sample Ratio Mismatch.

    Returns dict with chi2, p_value, and a plain-language verdict.
    SRM indicates randomisation bugs, bot traffic, or pipeline issues —
    if p < 0.01 do NOT trust downstream test results.
    """
    n_total = n_control + n_treatment
    expected = np.array([n_total * expected_ratio, n_total * (1 - expected_ratio)])
    observed = np.array([n_control, n_treatment])
    chi2, p = stats.chisquare(observed, f_exp=expected)
    verdict = (
        "SRM DETECTED — investigate randomisation"
        if p < 0.01
        else "OK — sample split consistent"
    )
    return {"chi2": float(chi2), "p_value": float(p), "verdict": verdict}


# ════════════════════════════════════════════════════════════════════════
# SMALL REUSABLE STATS
# ════════════════════════════════════════════════════════════════════════


def two_proportion_ztest(
    p_control: float, p_treatment: float, n_control: int, n_treatment: int
) -> tuple[float, float]:
    """Pooled two-proportion z-test. Returns (z_stat, two_sided_p_value)."""
    p_pool = (p_control * n_control + p_treatment * n_treatment) / (
        n_control + n_treatment
    )
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
    z = (p_treatment - p_control) / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p_value)


def cohens_h(p1: float, p2: float) -> float:
    """Cohen's h effect size for two proportions."""
    return float(2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1))))


def cohens_d(x1: np.ndarray, x2: np.ndarray) -> float:
    """Pooled Cohen's d effect size for two samples."""
    s_pool = np.sqrt((x1.var(ddof=1) + x2.var(ddof=1)) / 2)
    return float((x2.mean() - x1.mean()) / s_pool) if s_pool > 0 else 0.0


def interpret_magnitude(abs_effect: float) -> str:
    """Cohen convention: <0.2 negligible, <0.5 small, <0.8 medium, else large."""
    if abs_effect < 0.2:
        return "negligible"
    if abs_effect < 0.5:
        return "small"
    if abs_effect < 0.8:
        return "medium"
    return "large"


def print_header(title: str) -> None:
    """Consistent banner for each technique file."""
    print("=" * 70)
    print(f"  {title}")
    print("=" * 70)




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 3.2: Hypothesis Testing & Effect Sizes
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Run a two-proportion z-test for conversion rates
#   - Run Mann-Whitney U for skewed continuous metrics (revenue)
#   - Run Welch's t-test for continuous metrics (AOV, engagement)
#   - Compute and interpret Cohen's h (proportions) and Cohen's d (means)
#   - Distinguish statistical significance from practical significance
#   - Make a data-driven business decision
#
# PREREQUISITES: Exercise 3.1 (bootstrap CIs, power analysis)
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Load data and extract conversion/revenue arrays
#   2. Primary test: two-proportion z-test on conversion rate
#   3. Multiple metrics: revenue, AOV, engagement
#   4. Effect size interpretation (Cohen's h and d)
#   5. Visualise p-values and effect sizes
#   6. Business decision summary
#
# THEORY:
#   - Two-proportion z-test: H0: p_treatment = p_control
#     z = (p_t - p_c) / sqrt(p_pool * (1-p_pool) * (1/n1 + 1/n2))
#   - p-value = P(data | H0 true) -- NOT P(H0 is true)
#   - Cohen's h = 2(arcsin(sqrt(p2)) - arcsin(sqrt(p1)))
#   - Cohen's d = (mean2 - mean1) / s_pooled
#   - Convention: |effect| < 0.2 negligible, < 0.5 small, < 0.8 medium, else large
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from scipy import stats


print_header("MLFP02 Exercise 3.2: Hypothesis Testing & Effect Sizes")



## TASK 1 — Load data and extract arrays


In [ ]:
df = load_experiment()
control, treatment = split_groups(df)
n_control = control.height
n_treatment = treatment.height
n_total = df.height

ctrl_conv, treat_conv = conversion_arrays(df)
ctrl_rev, treat_rev = revenue_arrays(df)

p_control = ctrl_conv.mean()
p_treatment = treat_conv.mean()

print(f"Data loaded: {n_total:,} users")
print(f"  Control conversion:   {p_control:.4f} ({p_control:.2%})")
print(f"  Treatment conversion: {p_treatment:.4f} ({p_treatment:.2%})")



### Checkpoint 1


In [ ]:
assert 0 < p_control < 1, "Baseline must be a valid proportion"
assert 0 < p_treatment < 1, "Treatment must be a valid proportion"
print("\n>>> Checkpoint 1 passed -- data loaded\n")



## THEORY — What a p-value actually means


In [ ]:
# p-value = P(observing data this extreme OR more | H0 is true)
#
# It is NOT:
#   - P(H0 is true)
#   - P(the treatment works)
#   - The probability you are wrong
#
# A small p-value means the data is unlikely under the null hypothesis.
# A large effect with small n can have a large p-value (underpowered).
# A tiny effect with huge n can have a tiny p-value (overpowered).
#
# ALWAYS report both p-value AND effect size. p-value answers "is there
# a signal?" Effect size answers "is the signal large enough to matter?"



## TASK 2 — Primary test: two-proportion z-test


In [ ]:
# TODO: Call two_proportion_ztest(p_control, p_treatment, n_control, n_treatment).
# Hint: Returns (z_stat, p_value).
z_stat, p_value_conversion = ____

# TODO: Compute Cohen's h using cohens_h(p_control, p_treatment).
h_conversion = ____

# TODO: Interpret the magnitude using interpret_magnitude(abs(h_conversion)).
h_magnitude = ____

print(f"=== Primary Metric: Conversion Rate ===")
print(f"Control:   {p_control:.4f} ({p_control:.2%})")
print(f"Treatment: {p_treatment:.4f} ({p_treatment:.2%})")
print(f"Absolute lift: {p_treatment - p_control:+.4f}")
print(f"Relative lift: {(p_treatment - p_control) / p_control:+.2%}")
print(f"z-statistic: {z_stat:.4f}")
print(f"p-value: {p_value_conversion:.6f}")
print(f"Cohen's h: {h_conversion:.4f} ({h_magnitude})")
sig_label = "SIGNIFICANT" if p_value_conversion < ALPHA else "NOT significant"
print(f"{sig_label} at alpha={ALPHA}")

# INTERPRETATION: A small p-value means the observed difference is
# unlikely under H0 (no effect). It does NOT mean the effect is large
# or important -- a tiny effect with huge n can be highly significant.
# ALWAYS report effect size (Cohen's h) alongside the p-value.



### Checkpoint 2


In [ ]:
assert 0 <= p_value_conversion <= 1, "p-value must be between 0 and 1"
print("\n>>> Checkpoint 2 passed -- primary test completed\n")



## TASK 3 — Multiple metrics: revenue, AOV, engagement


In [ ]:
# Testing multiple metrics inflates Type I error.
# P(at least one false positive) = 1 - (1 - alpha)^m
# We collect ALL results here; corrections are in 03_multiple_testing.py.

metrics_results = {}

# Metric 1: Conversion (already computed)
metrics_results["conversion_rate"] = {
    "control": p_control,
    "treatment": p_treatment,
    "stat": z_stat,
    "p_value": p_value_conversion,
    "test": "z-test",
}

# Metric 2: Revenue per user (Mann-Whitney U for skewed data)
# TODO: Run stats.mannwhitneyu(treat_rev, ctrl_rev, alternative="two-sided").
# Hint: Returns (u_stat, p_value).
u_stat, p_value_revenue = ____

metrics_results["revenue_per_user"] = {
    "control": ctrl_rev.mean(),
    "treatment": treat_rev.mean(),
    "stat": u_stat,
    "p_value": p_value_revenue,
    "test": "Mann-Whitney U",
}

# Metric 3: Average order value (converters only, Welch's t-test)
aov_ctrl = (
    control.filter(pl.col("converted") == 1)["revenue"].to_numpy().astype(np.float64)
)
aov_treat = (
    treatment.filter(pl.col("converted") == 1)["revenue"].to_numpy().astype(np.float64)
)
if len(aov_ctrl) > 1 and len(aov_treat) > 1:
    # TODO: Run Welch's t-test: stats.ttest_ind(aov_treat, aov_ctrl, equal_var=False)
    t_aov, p_aov = ____
    metrics_results["avg_order_value"] = {
        "control": aov_ctrl.mean(),
        "treatment": aov_treat.mean(),
        "stat": t_aov,
        "p_value": p_aov,
        "test": "Welch's t-test",
    }

# Metric 4: Primary engagement metric (metric_value)
mv_ctrl = control["metric_value"].to_numpy().astype(np.float64)
mv_treat = treatment["metric_value"].to_numpy().astype(np.float64)
t_mv, p_mv = stats.ttest_ind(mv_treat, mv_ctrl, equal_var=False)
metrics_results["metric_value"] = {
    "control": mv_ctrl.mean(),
    "treatment": mv_treat.mean(),
    "stat": t_mv,
    "p_value": p_mv,
    "test": "Welch's t-test",
}

# Metric 5: Pages viewed (if available)
if "pages_viewed" in df.columns:
    pg_ctrl = control["pages_viewed"].to_numpy().astype(np.float64)
    pg_treat = treatment["pages_viewed"].to_numpy().astype(np.float64)
    t_pg, p_pg = stats.ttest_ind(pg_treat, pg_ctrl, equal_var=False)
    metrics_results["pages_viewed"] = {
        "control": pg_ctrl.mean(),
        "treatment": pg_treat.mean(),
        "stat": t_pg,
        "p_value": p_pg,
        "test": "Welch's t-test",
    }

m = len(metrics_results)

# TODO: Compute the family-wise error rate: 1 - (1 - ALPHA) ** m
fwer = ____

print(f"=== Multiple Metric Results (unadjusted) ===")
print(
    f"{'Metric':<20} {'Control':>12} {'Treatment':>12} "
    f"{'p-value':>10} {'Test':<18} {'Sig':>4}"
)
print("-" * 80)
for name, r in metrics_results.items():
    sig = (
        "***"
        if r["p_value"] < 0.001
        else "**" if r["p_value"] < 0.01 else "*" if r["p_value"] < 0.05 else "ns"
    )
    print(
        f"{name:<20} {r['control']:>12.4f} {r['treatment']:>12.4f} "
        f"{r['p_value']:>10.6f} {r['test']:<18} {sig:>4}"
    )

print(f"\nNumber of tests (m): {m}")
print(f"Family-wise error rate (uncorrected): {fwer:.4f} ({fwer:.1%})")



### Checkpoint 3


In [ ]:
assert m >= 3, "Should test at least 3 metrics"
assert fwer > ALPHA, "FWER must exceed single-test alpha"
print("\n>>> Checkpoint 3 passed -- multiple metrics tested\n")



## TASK 4 — Effect size interpretation


In [ ]:
# Statistical significance != practical significance.
# Cohen's h (proportions): small=0.2, medium=0.5, large=0.8
# Cohen's d (means): small=0.2, medium=0.5, large=0.8

print(f"=== Effect Size Interpretation ===")

for name, r in metrics_results.items():
    ctrl_val = r["control"]
    treat_val = r["treatment"]

    if name == "conversion_rate":
        # TODO: Compute Cohen's h using cohens_h(ctrl_val, treat_val)
        effect = ____
        metric_label = "Cohen's h"
    else:
        # TODO: Compute Cohen's d using cohens_d with the appropriate arrays.
        # Hint: For revenue use cohens_d(ctrl_rev, treat_rev),
        #       for metric_value use cohens_d(mv_ctrl, mv_treat),
        #       for AOV use cohens_d(aov_ctrl, aov_treat).
        if name == "revenue_per_user":
            effect = ____
        elif name == "metric_value":
            effect = ____
        elif name == "avg_order_value" and len(aov_ctrl) > 1:
            effect = ____
        else:
            effect = 0.0
        metric_label = "Cohen's d"

    magnitude = interpret_magnitude(abs(effect))
    sig_str = "sig" if r["p_value"] < ALPHA else "ns"
    print(
        f"  {name:<20}: {metric_label}={effect:+.4f} ({magnitude}) "
        f"| p={r['p_value']:.4f} ({sig_str})"
    )

print(f"\n--- Key Insight ---")
print(f"A metric can be statistically significant but practically negligible.")
print(
    f"A metric can be practically important but statistically "
    f"non-significant (underpowered)."
)
print(f"Always report BOTH p-value AND effect size for informed decision-making.")



### Checkpoint 4


In [ ]:
assert h_conversion is not None, "Cohen's h must be computed"
print("\n>>> Checkpoint 4 passed -- effect sizes computed and interpreted\n")



## TASK 5 — Visualise: p-value bar chart


In [ ]:
metric_names = list(metrics_results.keys())
p_values = np.array([r["p_value"] for r in metrics_results.values()])
bonferroni_alpha = ALPHA / m

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=metric_names,
        y=p_values,
        name="Raw p-value",
        marker_color="lightblue",
    )
)
fig.add_hline(y=ALPHA, line_dash="dash", annotation_text=f"alpha={ALPHA}")
fig.add_hline(
    y=bonferroni_alpha,
    line_dash="dot",
    line_color="red",
    annotation_text=f"Bonferroni alpha={bonferroni_alpha:.4f}",
)
fig.update_layout(
    title="P-values with Multiple Testing Thresholds",
    yaxis_title="p-value",
    yaxis_type="log",
)
out_path = OUTPUT_DIR / "pvalue_comparison.html"
fig.write_html(str(out_path))
print(f"Saved: {out_path}")



## APPLY — Business decision for Singapore e-commerce


Experiment: e-commerce recommendation algorithm A/B test
Sample: {n_total:,} users ({n_control:,} control, {n_treatment:,} treatment)

Primary metric (conversion):
  Lift: {p_treatment - p_control:+.4f} ({(p_treatment - p_control)/p_control:+.1%} relative)
  p-value: {p_value_conversion:.6f} | Cohen's h: {h_conversion:.4f} ({h_magnitude})

Multiple metrics tested: {m} (corrections needed -- see 03_multiple_testing.py)
  FWER without correction: {fwer:.1%}

Recommendation: {decision}


In [ ]:
# SCENARIO: The data science team at a Singapore e-commerce company has
# run a 2-week A/B test on a new recommendation algorithm. The product
# manager asks: "Should we ship the treatment?"
#
# The answer depends on BOTH the p-value and the effect size:
#   - p < 0.05 AND |Cohen's h| > 0.1 -> Ship with confidence
#   - p < 0.05 AND |Cohen's h| < 0.1 -> Significant but negligible
#   - p > 0.05 AND |Cohen's h| > 0.1 -> Likely underpowered, run longer
#   - p > 0.05 AND |Cohen's h| < 0.1 -> No meaningful effect
#
# BUSINESS IMPACT: For a platform processing S$10M daily GMV, a 2pp
# conversion lift means ~S$200K additional daily revenue. Even a
# "small" Cohen's h of 0.05 translates to real money at scale.

print(f"\n=== Business Decision Summary ===")
decision = (
    "Ship the treatment"
    if p_value_conversion < ALPHA and abs(h_conversion) > 0.1
    else (
        "Need more data"
        if abs(h_conversion) > 0.05
        else "No meaningful effect detected"
    )
)
print(
)



## REFLECTION


[x] Two-proportion z-test for conversion rates
  [x] Mann-Whitney U for skewed continuous metrics (revenue)
  [x] Welch's t-test for continuous metrics (AOV, engagement)
  [x] p-value: P(data | H0 true) -- NOT P(H0 is true)
  [x] Cohen's h and d -- practical vs statistical significance
  [x] Business decision framework: both p-value AND effect size

  NEXT: In 03_multiple_testing.py you'll apply Bonferroni and BH-FDR
  corrections to control error rates across multiple metrics.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print(">>> Exercise 3.2 complete -- Hypothesis Testing & Effect Sizes")

